In [2]:
# ==========================================
# 0. PREPARAÇÃO DO CSV
# ==========================================
import pandas as pd
import json
import os

# Instalando a biblioteca da OpenAI
!pip install -q openai
from openai import OpenAI

# Criando um CSV fictício diretamente no Colab
dados_ficticios = {
    'UserID': [1, 2, 3],
    'Name': ['Ana', 'Carlos', 'Beatriz']
}
df_ficticio = pd.DataFrame(dados_ficticios)
df_ficticio.to_csv('SDW2023_ficticio.csv', index=False)
print("✅ Arquivo CSV fictício 'SDW2023_ficticio.csv' criado com sucesso!\n")


# ==========================================
# 1. EXTRACT (Extração)
# ==========================================
# Lendo o CSV criado e transformando em uma lista de dicionários
df_extraido = pd.read_csv('SDW2023_ficticio.csv')
users = df_extraido.to_dict(orient='records')

# Ajustando a estrutura para ficar parecida com a que a API original retornava
for user in users:
    user['id'] = user.pop('UserID')  # Renomeia UserID para id
    user['name'] = user.pop('Name')  # Renomeia Name para name
    user['news'] = []                # Cria a lista vazia de notícias

print("✅ Dados extraídos e estruturados:")
print(json.dumps(users, indent=2, ensure_ascii=False))
print("-" * 50)


# ==========================================
# 2. TRANSFORM (Transformação)
# ==========================================
#Chave da OpenAI
OPENAI_API_KEY = 'CHAVE'

# Configurando o cliente da OpenAI (sintaxe moderna da biblioteca)
client = OpenAI(api_key=OPENAI_API_KEY)

def generate_ai_news(user):
    # Se a chave não for preenchida, retorna uma mensagem padrão para não dar erro
    if OPENAI_API_KEY == 'CHAVE' or OPENAI_API_KEY == 'TODO':
        return f"{user['name']}, invista hoje para um amanhã seguro. Seu dinheiro pode render muito mais!"

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {
                    "role": "system",
                    "content": "Você é um especialista em marketing bancário."
                },
                {
                    "role": "user",
                    "content": f"Crie uma mensagem para {user['name']} sobre a importância dos investimentos (máximo de 100 caracteres)"
                }
            ]
        )
        return response.choices[0].message.content.strip('\"')
    except Exception as e:
        return f"Erro ao acessar API: {e}"

# Aplicação a transformação para cada usuário
print("🤖 Gerando mensagens de marketing...")
for user in users:
    news = generate_ai_news(user)
    print(f"Mensagem para {user['name']}: {news}")

    user['news'].append({
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": news
    })
print("-" * 50)


# ==========================================
# 3. LOAD (Carregamento)
# ==========================================
# Como a API original está fora do ar (não será possivel fazer o PUT),
# O Save será em um novo arquivo JSON local para simular a carga.

arquivo_saida = 'SDW2023_Atualizado.json'

with open(arquivo_saida, 'w', encoding='utf-8') as f:
    json.dump(users, f, indent=4, ensure_ascii=False)

print(f"✅ Processo LOAD concluído! Os dados atualizados foram salvos em '{arquivo_saida}'.")
print("Visualização final dos dados:")
print(json.dumps(users, indent=2, ensure_ascii=False))

✅ Arquivo CSV fictício 'SDW2023_ficticio.csv' criado com sucesso!

✅ Dados extraídos e estruturados:
[
  {
    "id": 1,
    "name": "Ana",
    "news": []
  },
  {
    "id": 2,
    "name": "Carlos",
    "news": []
  },
  {
    "id": 3,
    "name": "Beatriz",
    "news": []
  }
]
--------------------------------------------------
🤖 Gerando mensagens de marketing...
Mensagem para Ana: Ana, invista hoje para um amanhã seguro. Seu dinheiro pode render muito mais!
Mensagem para Carlos: Carlos, invista hoje para um amanhã seguro. Seu dinheiro pode render muito mais!
Mensagem para Beatriz: Beatriz, invista hoje para um amanhã seguro. Seu dinheiro pode render muito mais!
--------------------------------------------------
✅ Processo LOAD concluído! Os dados atualizados foram salvos em 'SDW2023_Atualizado.json'.
Visualização final dos dados:
[
  {
    "id": 1,
    "name": "Ana",
    "news": [
      {
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.